# Fase 4 · Visualización y comunicación de resultados
## Licitaciones públicas de ChileCompra · MCDI500 · Grupo 2

Este cuaderno prepara y verifica los insumos descriptivos de la Fase 4 a partir de las salidas consolidadas en `data/processed/`, respondiendo las preguntas definidas en `F1/F1_Definicion.ipynb` (sección 3):

1. **Coincidencia y brecha por moneda:** en qué proporción de las licitaciones comparables coincide el monto estimado registrado con el total de las ofertas ganadoras y, cuando difieren, cuál es el sentido y la magnitud de la brecha en cada moneda (`CLP`, `CLF`, `USD`, `UTM`).
2. **Comparación según tamaño del proveedor ganador:** cómo cambian la coincidencia y la brecha según `TamanoProveedorOrdinal` (`Micro < Pequeña < Mediana < Grande`, manteniendo `NoClasificado` fuera de la escala).

**Flujo del cuaderno (F1–F4).** Siguiendo `docs/arquitectura.md`, la lógica de indicadores y gráficos reside en **`src/comunicacion.py`**. Este cuaderno implementa la preparación verificable previa al graficado:
1. **Entorno, configuración y estilo de figuras** (`sys.prefix`, `requirements.txt` y `plt.rcParams` definido una vez).
2. **Carga de artefactos y contrato de verificación** (`cargar_artefactos` con `sep=";"` y `usecols`, y `assert` contra `diccionario_f3.csv` y `parametros_f3.csv`).
3. **Verificación independiente contra `data/raw`** (recálculo directo desde el CSV original sobre una muestra diversa de licitaciones).
4. **Copia interpretable para graficar y vinculación Objetivo · Pregunta · Figura** (etiquetas, tramos, brecha relativa y `NoClasificado` fuera del eje ordinal, más tabla de vinculación y exportación de `docs/metadatos_f4.json`).

**Cómo ejecutar.** Genera primero las referencias en `data/processed/` (ejecutando `F2` y `F3`) y ejecuta todas las celdas con el kernel **Python (grupo2-mcdi500)**.


## 1. Entorno, configuración y estilo de figuras

Comprueba que el cuaderno se ejecuta dentro del entorno virtual `.venv` mediante `sys.prefix`, valida las versiones declaradas en `requirements.txt` (`pandas`, `numpy`, `matplotlib`), importa `src/comunicacion.py`, verifica la presencia de los cuatro artefactos en `data/processed/` y fija **una única vez** el estilo global de figuras (`plt.rcParams`).


In [1]:
# 1) Entorno con sys.prefix, configuración de rutas y estilo global de figuras.
import json
import math
import platform
import sys
from importlib import metadata
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PAQUETES_F4 = ["pandas", "numpy", "matplotlib"]

inicio = Path.cwd().resolve()
raiz = next((p for p in (inicio, *inicio.parents) if (p / "requirements.txt").is_file()), None)
if raiz is None:
    raise FileNotFoundError("No se encontró requirements.txt: abre el notebook dentro del repositorio.")
if sys.prefix == sys.base_prefix:
    raise RuntimeError("No se usa un entorno virtual: cambia el kernel a 'Python (grupo2-mcdi500)'.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.comunicacion import (
    ARCHIVOS_PROCESADOS,
    COLUMNAS_LICITACIONES_F4,
    COLUMNAS_OFERTAS_F4,
    ORDEN_SENTIDO,
    asignar_tamano_ganador,
    cargar_artefactos,
    preparar_comparables,
)
from src.preprocesamiento import ORDEN_TAMANO, normalizar_codigo

declaradas = {}
for linea in (raiz / "requirements.txt").read_text(encoding="utf-8").splitlines():
    nombre, _, ver = linea.strip().partition("==")
    if nombre.lower() in PAQUETES_F4:
        declaradas[nombre.lower()] = ver

versiones_f4 = pd.DataFrame([
    {
        "Paquete": pkg,
        "Declarada": declaradas.get(pkg),
        "Instalada": metadata.version(pkg),
        "Coincide": metadata.version(pkg) == declaradas.get(pkg),
    }
    for pkg in PAQUETES_F4
])

ESTILO_FIGURAS_F4 = {
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
    "legend.frameon": True,
}
plt.rcParams.update(ESTILO_FIGURAS_F4)

carpeta_procesados = raiz / "data/processed"
carpeta_figuras = raiz / "F4/figuras"
ruta_raw = raiz / "data/raw/04LicitacionesEne.csv"
for clave, archivo in ARCHIVOS_PROCESADOS.items():
    ruta_art = carpeta_procesados / archivo
    if not ruta_art.is_file():
        raise FileNotFoundError(f"Falta {ruta_art.relative_to(raiz)}. Ejecuta F2 y F3 primero.")

print(f"Intérprete: {Path(sys.executable).relative_to(raiz)} | Python {platform.python_version()} | {platform.system()}")
print(f"sys.prefix      : {sys.prefix}")
print(f"sys.base_prefix : {sys.base_prefix}")
print("[OK] Entorno virtual activo y estilo global plt.rcParams definido una vez.")
display(versiones_f4)


Intérprete: .venv\Scripts\python.exe | Python 3.15.2 | Windows
sys.prefix      : C:\Users\danie\Desktop\proyecto-grupo2-mcdi500\.venv
sys.base_prefix : C:\Program Files\Python315
[OK] Entorno virtual activo y estilo global plt.rcParams definido una vez.


,Paquete,Declarada,Instalada,Coincide
0,pandas,3.1.0,3.1.0,True
1,numpy,2.5.4,2.5.4,True
2,matplotlib,3.11.2,3.11.2,True


## 2. Carga de artefactos y contrato con `assert`

Carga los cuatro archivos de `data/processed/` con `cargar_artefactos()` (`sep=";"` y `usecols`), tipa explícitamente las columnas booleanas con valores faltantes (`EsExtremoLogIQR_CLP` y `CoincideConEstimado`) y verifica por contrato (`assert`) contra `diccionario_f3.csv` y `parametros_f3.csv`:

- **Esquema:** coincidencia exacta entre los encabezados de cada archivo CSV (`17` y `24` columnas) y las `41` entradas de `diccionario_f3.csv`.
- **Cardinalidad e integridad:** `45.741` ofertas, `2.292` licitaciones únicas (`NroLicitacion`), mismo conjunto de códigos en ambas tablas y consistencia con `ofertas_validas = 45.603` de `parametros_f3.csv`.
- **Tipos de datos:** compatibilidad de cada columna cargada frente al `Tipo en Python` declarado en `diccionario_f3.csv`.


In [2]:
# 2) Carga selectiva (usecols) de artefactos y validación de contrato contra diccionario_f3 y parametros_f3.
artefactos = cargar_artefactos(carpeta_procesados)
ofertas_f4 = artefactos.ofertas.copy()
licitaciones_f4 = artefactos.licitaciones.copy()
diccionario = artefactos.diccionario
parametros = artefactos.parametros

# Restaurar tipo nullable boolean de diccionario_f3 para columnas con <NA>.
ofertas_f4["EsExtremoLogIQR_CLP"] = ofertas_f4["EsExtremoLogIQR_CLP"].astype("boolean")
licitaciones_f4["CoincideConEstimado"] = licitaciones_f4["CoincideConEstimado"].astype("boolean")

# 2.1 Contrato de esquema completo (archivos vs diccionario_f3.csv).
cols_archivo_ofertas = pd.read_csv(carpeta_procesados / ARCHIVOS_PROCESADOS["ofertas"], sep=";", encoding="utf-8-sig", nrows=0).columns.tolist()
cols_archivo_lic = pd.read_csv(carpeta_procesados / ARCHIVOS_PROCESADOS["licitaciones"], sep=";", encoding="utf-8-sig", nrows=0).columns.tolist()
dic_ofertas = diccionario.loc[diccionario["Tabla"].eq("ofertas")].set_index("Columna")
dic_lic = diccionario.loc[diccionario["Tabla"].eq("licitaciones")].set_index("Columna")

assert set(cols_archivo_ofertas) == set(dic_ofertas.index), "Desajuste de esquema en ofertas vs diccionario_f3"
assert set(cols_archivo_lic) == set(dic_lic.index), "Desajuste de esquema en licitaciones vs diccionario_f3"
assert set(COLUMNAS_OFERTAS_F4) == set(ofertas_f4.columns)
assert set(COLUMNAS_LICITACIONES_F4) == set(licitaciones_f4.columns)
assert len(diccionario) == len(cols_archivo_ofertas) + len(cols_archivo_lic) == 41

# 2.2 Contrato de filas e integridad referencial.
assert len(ofertas_f4) == 45741, f"Se esperaban 45.741 ofertas y hay {len(ofertas_f4)}"
assert len(licitaciones_f4) == 2292, f"Se esperaban 2.292 licitaciones y hay {len(licitaciones_f4)}"
assert licitaciones_f4["NroLicitacion"].is_unique, "NroLicitacion repetido en licitaciones"
assert set(normalizar_codigo(ofertas_f4["NroLicitacion"]).dropna()) == set(normalizar_codigo(licitaciones_f4["NroLicitacion"]))
ofertas_clp_param = int(float(parametros.loc[parametros["Parámetro"].eq("ofertas_validas"), "Valor"].iloc[0]))
assert int(ofertas_f4["EsExtremoLogIQR_CLP"].notna().sum()) == ofertas_clp_param == 45603

# 2.3 Contrato de tipos frente a diccionario_f3['Tipo en Python'].
def _tipo_compatible(serie: pd.Series, tipo_dic: str) -> bool:
    if tipo_dic in {"Float64", "Int64", "int64"}:
        return pd.api.types.is_numeric_dtype(serie.dtype)
    if tipo_dic in {"boolean", "bool"}:
        return pd.api.types.is_bool_dtype(serie.dtype)
    if tipo_dic == "category":
        return isinstance(serie.dtype, pd.CategoricalDtype) and serie.cat.ordered and tuple(serie.cat.categories) == ORDEN_TAMANO
    if tipo_dic in {"string", "str"}:
        return pd.api.types.is_string_dtype(serie.dtype)
    return False

for col in ofertas_f4.columns:
    tipo_esp = str(dic_ofertas.loc[col, "Tipo en Python"])
    assert _tipo_compatible(ofertas_f4[col], tipo_esp), f"Tipo incompatible en ofertas[{col}]: {ofertas_f4[col].dtype} vs {tipo_esp}"
for col in licitaciones_f4.columns:
    tipo_esp = str(dic_lic.loc[col, "Tipo en Python"])
    assert _tipo_compatible(licitaciones_f4[col], tipo_esp), f"Tipo incompatible en licitaciones[{col}]: {licitaciones_f4[col].dtype} vs {tipo_esp}"

resumen_contrato = pd.DataFrame([
    {
        "Tabla": "ofertas",
        "Filas": len(ofertas_f4),
        "Columnas archivo": len(cols_archivo_ofertas),
        "Columnas usecols": ofertas_f4.shape[1],
        "Esquema diccionario_f3": "OK",
        "Tipos": "OK",
    },
    {
        "Tabla": "licitaciones",
        "Filas": len(licitaciones_f4),
        "Columnas archivo": len(cols_archivo_lic),
        "Columnas usecols": licitaciones_f4.shape[1],
        "Esquema diccionario_f3": "OK",
        "Tipos": "OK",
    },
])
print("[OK] Contrato de esquema (41 columnas), cardinalidad (45.741 / 2.292), parámetros (45.603 CLP) y tipos verificado.")
display(resumen_contrato)


[OK] Contrato de esquema (41 columnas), cardinalidad (45.741 / 2.292), parámetros (45.603 CLP) y tipos verificado.


,Tabla,Filas,Columnas archivo,Columnas usecols,Esquema diccionario_f3,Tipos
0,ofertas,45741,17,7,OK,OK
1,licitaciones,2292,24,11,OK,OK


## 3. Verificación independiente contra `data/raw`

Para descartar errores de arrastre entre fases, si el archivo original `data/raw/04LicitacionesEne.csv` está disponible localmente, se lee directamente con `sep=";"` y `encoding="latin-1"` de manera independiente a `src/comunicacion.py` y se recalcula desde cero:

- Cantidad total de filas (`45.741`) y licitaciones únicas (`2.292`).
- **En una muestra determinista de `7` licitaciones comparables** (que abarca coincidencia exacta, brecha negativa, brecha positiva, múltiples ofertas ganadoras —desde `1` hasta `114` ítems— y las monedas `CLP`, `CLF` y `USD`): cantidad de ganadoras, `MontoTotalGanadoras` sumado desde las filas crudas (con conversión directa de coma decimal) y `BrechaMonto`, comprobando con `assert` su coincidencia frente a `artefactos.licitaciones`.


In [3]:
# 3) Verificación independiente contra data/raw recalculando una muestra desde el CSV original.
verificacion_raw = {"disponible": bool(ruta_raw.is_file()), "muestra": 0, "estado": "OMITIDA (sin CSV crudo local)"}

if ruta_raw.is_file():
    crudo = pd.read_csv(
        ruta_raw,
        sep=";",
        encoding="latin-1",
        usecols=["NroLicitacion", "ResultadoOferta", "MontoTotalOferta", "MontoEstimadoLicitacion", "MonedaLicitacion"],
        dtype="string",
    )
    assert len(crudo) == len(ofertas_f4) == 45741
    crudo["Nro_norm"] = crudo["NroLicitacion"].str.strip().str.upper()
    assert crudo["Nro_norm"].nunique() == len(licitaciones_f4) == 2292

    comp_proc = licitaciones_f4.loc[licitaciones_f4["EstadoBrecha"].eq("Calculada")].copy()
    candidatos = [
        comp_proc.loc[comp_proc["CoincideConEstimado"].fillna(False) & comp_proc["OfertasGanadoras"].eq(1), "NroLicitacion"].iloc[0],
        comp_proc.loc[comp_proc["CoincideConEstimado"].fillna(False) & comp_proc["OfertasGanadoras"].gt(1), "NroLicitacion"].iloc[0],
        comp_proc.loc[comp_proc["BrechaMonto"].lt(0) & comp_proc["MonedaLicitacion"].eq("CLP"), "NroLicitacion"].iloc[0],
        comp_proc.loc[comp_proc["BrechaMonto"].gt(0) & comp_proc["MonedaLicitacion"].eq("CLP"), "NroLicitacion"].iloc[0],
        comp_proc.loc[comp_proc["BrechaMonto"].gt(0) & comp_proc["OfertasGanadoras"].gt(3), "NroLicitacion"].iloc[0],
        comp_proc.loc[comp_proc["BrechaMonto"].lt(0) & comp_proc["OfertasGanadoras"].gt(3), "NroLicitacion"].iloc[0],
        comp_proc.loc[comp_proc["MonedaLicitacion"].eq("CLF"), "NroLicitacion"].iloc[0],
        comp_proc.loc[comp_proc["MonedaLicitacion"].eq("USD"), "NroLicitacion"].iloc[0],
    ]
    muestra_ids = list(dict.fromkeys(candidatos))

    sub_raw = crudo.loc[crudo["Nro_norm"].isin(muestra_ids)].copy()
    sub_raw["MontoTotalNum"] = pd.to_numeric(sub_raw["MontoTotalOferta"].str.strip().str.replace(",", ".", regex=False), errors="raise")
    sub_raw["EstimadoNum"] = pd.to_numeric(sub_raw["MontoEstimadoLicitacion"].str.strip().str.replace(",", ".", regex=False), errors="raise")
    sub_raw["EsGanadora"] = sub_raw["ResultadoOferta"].str.strip().str.casefold().eq("ganadora")

    filas_verif = []
    proc_idx = comp_proc.set_index("NroLicitacion")
    for lic_id in muestra_ids:
        g_raw = sub_raw.loc[sub_raw["Nro_norm"].eq(lic_id)]
        gan_raw = g_raw.loc[g_raw["EsGanadora"]]
        total_gan_raw = float(gan_raw["MontoTotalNum"].sum())
        est_raw = float(g_raw["EstimadoNum"].iloc[0])
        brecha_raw = total_gan_raw - est_raw
        n_gan_raw = int(len(gan_raw))

        fila_proc = proc_idx.loc[lic_id]
        assert n_gan_raw == int(fila_proc["OfertasGanadoras"]), f"Diferencia en ganadoras para {lic_id}"
        assert math.isclose(total_gan_raw, float(fila_proc["MontoTotalGanadoras"]), rel_tol=1e-12, abs_tol=1e-6)
        assert math.isclose(brecha_raw, float(fila_proc["BrechaMonto"]), rel_tol=1e-12, abs_tol=1e-6)

        filas_verif.append({
            "NroLicitacion": lic_id,
            "Moneda": fila_proc["MonedaLicitacion"],
            "Ganadoras_raw": n_gan_raw,
            "TotalGanadoras_raw": total_gan_raw,
            "TotalGanadoras_proc": float(fila_proc["MontoTotalGanadoras"]),
            "Brecha_raw": brecha_raw,
            "Brecha_proc": float(fila_proc["BrechaMonto"]),
            "Diferencia_abs": abs(brecha_raw - float(fila_proc["BrechaMonto"])),
        })

    tabla_verif_raw = pd.DataFrame(filas_verif)
    verificacion_raw = {
        "disponible": True,
        "filas_crudo": int(len(crudo)),
        "licitaciones_unicas_crudo": int(crudo["Nro_norm"].nunique()),
        "muestra_licitaciones": len(tabla_verif_raw),
        "max_diferencia_abs": float(tabla_verif_raw["Diferencia_abs"].max()),
        "estado": "APROBADA",
    }
    print(f"[OK] Verificación contra data/raw aprobada sobre {len(tabla_verif_raw)} licitaciones (máxima diferencia={verificacion_raw['max_diferencia_abs']:.1e}).")
    display(tabla_verif_raw.round(4))
else:
    print("[AVISO] No se encontró data/raw/04LicitacionesEne.csv localmente; se omite la verificación contra el CSV original.")


[OK] Verificación contra data/raw aprobada sobre 7 licitaciones (máxima diferencia=0.0e+00).


,NroLicitacion,Moneda,Ganadoras_raw,TotalGanadoras_raw,TotalGanadoras_proc,Brecha_raw,Brecha_proc,Diferencia_abs
0,4080-31-LP24,CLP,1,56317830.0,56317830.0,0.0,0.0,0.0
1,2366-168-LE24,CLP,9,13210200.0,13210200.0,0.0,0.0,0.0
2,2410-128-LP24,CLP,2,2.0,2.0,-71499998.0,-71499998.0,0.0
3,2366-129-L124,CLP,16,11758244.0,11758244.0,8127774.0,8127774.0,0.0
4,2709-190-LR24,CLP,114,7532080.0,7532080.0,-62299448.0,-62299448.0,0.0
5,2410-201-LQ24,CLF,1,4140.0,4140.0,-787.0,-787.0,0.0
6,584146-1-LP25,USD,1,89376.0,89376.0,0.0,0.0,0.0


## 4. Copia interpretable para graficar y vinculación con objetivos

En flujos donde el preprocesamiento estandariza o codifica numéricamente las variables, suele requerirse separar una matriz transformada de una copia interpretable con unidades y etiquetas originales. En este proyecto, **F2 y F3 no aplicaron escalamiento ni codificación entera**: conservaron los montos en su moneda original (`CLP`, `CLF`, `USD`, `UTM`) y las categorías en texto (`TamanoProveedor` y `TamanoProveedorOrdinal`). Por ello **no aplican** funciones como `revertir_escalamiento` ni `reconstruir_categorias` (decisión registrada en `docs/decisiones.md`).

La copia `copia_graficar` parte de las tablas ya preparadas por `src/comunicacion.py` (`preparar_comparables()` y `asignar_tamano_ganador()`) y **únicamente añade atributos interpretables para visualización**:

1. **Etiquetas legibles:** `EtiquetaCoincidencia` (`"Coincide"` frente a `"Difiere"`), `SentidoBrecha` (`"Bajo el estimado"`, `"Coincide"`, `"Sobre el estimado"`) y `TamanoGanador` (incluyendo `"Múltiple"` cuando concurren varios tamaños ganadores).
2. **Tramos de monto estimado (`TramoEstimado`):** cuatro rangos legibles en `CLP` (`Hasta 1 M CLP`, `1 a 10 M CLP`, `10 a 50 M CLP`, `Más de 50 M CLP`) y la etiqueta explícita `"Otra moneda (no CLP)"` para no mezclar unidades.
3. **Brecha relativa (`BrechaRelativaPct`):** `(BrechaMonto / MontoEstimadoLicitacion) * 100`, complementando `RazonMonto` y `LogRazonMonto`.
4. **`NoClasificado` fuera del eje ordinal:** `TamanoGanadorOrdinal` conserva estrictamente `Micro < Pequeña < Mediana < Grande`; tanto `NoClasificado` como `Múltiple` permanecen como `<NA>` en el eje ordinal y se reportan aparte en `TamanoGanador`.


In [4]:
# 4) Construir copia interpretable para graficar desde src/comunicacion.py (sin revertir escalamiento).
comparables = asignar_tamano_ganador(
    preparar_comparables(licitaciones_f4),
    ofertas_f4,
)

etiqueta_coinc = comparables["CoincideConEstimado"].map({True: "Coincide", False: "Difiere"})
brecha_rel_pct = (comparables["RazonMonto"] - 1.0) * 100.0

es_clp = comparables["MonedaLicitacion"].eq("CLP")
tramos_clp = pd.cut(
    comparables.loc[es_clp, "MontoEstimadoLicitacion"],
    bins=[0.0, 1_000_000.0, 10_000_000.0, 50_000_000.0, float("inf")],
    labels=["Hasta 1 M CLP", "1 a 10 M CLP", "10 a 50 M CLP", "Más de 50 M CLP"],
).astype(str)
tramo_estimado = pd.Series("Otra moneda (no CLP)", index=comparables.index, dtype="string")
tramo_estimado.loc[es_clp] = tramos_clp
orden_tramos = ["Hasta 1 M CLP", "1 a 10 M CLP", "10 a 50 M CLP", "Más de 50 M CLP", "Otra moneda (no CLP)"]
tramo_estimado = pd.Categorical(tramo_estimado, categories=orden_tramos, ordered=True)

copia_graficar = comparables.assign(
    EtiquetaCoincidencia=etiqueta_coinc,
    BrechaRelativaPct=brecha_rel_pct,
    TramoEstimado=tramo_estimado,
)

# Verificar invariantes de la copia para graficar.
fuera_ordinal = copia_graficar["TamanoGanador"].isin(["NoClasificado", "Múltiple"])
assert copia_graficar.loc[fuera_ordinal, "TamanoGanadorOrdinal"].isna().all(), "NoClasificado/Múltiple no deben entrar al eje ordinal"
assert tuple(copia_graficar["TamanoGanadorOrdinal"].cat.categories) == ORDEN_TAMANO
assert copia_graficar["BrechaRelativaPct"].notna().all()

print(f"[OK] copia_graficar construida: {copia_graficar.shape[0]:,} licitaciones × {copia_graficar.shape[1]} columnas")
display(copia_graficar["TramoEstimado"].value_counts(sort=False).to_frame("Licitaciones"))
display(copia_graficar.groupby(["TamanoGanador", "TamanoGanadorOrdinal"], dropna=False, observed=False).size().loc[lambda s: s > 0].to_frame("Licitaciones"))


[OK] copia_graficar construida: 2,257 licitaciones × 19 columnas


,Licitaciones
TramoEstimado,
Hasta 1 M CLP,350
1 a 10 M CLP,739
10 a 50 M CLP,783
Más de 50 M CLP,358
Otra moneda (no CLP),27


,,Licitaciones
TamanoGanador,TamanoGanadorOrdinal,
Grande,Grande,365
Mediana,Mediana,273
Micro,Micro,405
Múltiple,nan,240
NoClasificado,nan,284
Pequeña,Pequeña,690


### 4.1. Tabla de vinculación: objetivo, pregunta y figura planificada

Antes de generar los gráficos de comunicación en la siguiente etapa de F4, cada figura queda trazada explícitamente hacia un objetivo específico y una pregunta de investigación de `F1/F1_Definicion.ipynb`:

| Objetivo específico (F1) | Pregunta de investigación (F1) | Figura / salida planificada (F4) | Función en `src/comunicacion.py` |
| :--- | :--- | :--- | :--- |
| **1.** Determinar en qué proporción de las licitaciones comparables coincide el monto estimado registrado con el total ganador | **P1 (parte 1):** ¿En qué proporción de las licitaciones comparables coincide el estimado con el total ganador? | Tasa de coincidencia global y por moneda con IC `95 %` de Wilson (`01_coincidencia_y_sentido.png`) | `intervalo_wilson()`, `resumen_sentido_por_moneda()`, `graficar_coincidencia_y_sentido()` |
| **2.** Describir la magnitud y el sentido de la brecha cuando difieren, separando por moneda | **P1 (parte 2):** Cuando difieren, ¿cuál es la magnitud y el sentido de la brecha en cada moneda? | Barras apiladas de `SentidoBrecha` (`01_coincidencia_y_sentido.png`) y distribución de `LogRazonMonto` con mediana/IQR en `CLP` (`02_distribucion_log_razon_clp.png`), más tabla en moneda original | `resumen_magnitud_por_moneda()`, `graficar_distribucion_brecha()` |
| **3.** Comparar la coincidencia y la brecha según el tamaño del proveedor ganador | **P2:** ¿Cambian la coincidencia y la brecha según el tamaño (`Micro`, `Pequeña`, `Mediana`, `Grande`)? | Tasa de coincidencia (IC Wilson) e intervalo `[Q1, Q3]` de `LogRazonMonto` por tamaño en `CLP`, dejando `NoClasificado` y `Múltiple` aparte (`03_resultados_por_tamano_clp.png`) | `tabla_resumen_por_tamano()`, `graficar_resultados_por_tamano()` |


## 5. Registro de metadatos de Fase 4 (`docs/metadatos_f4.json`)

Consolida en [`docs/metadatos_f4.json`](../docs/metadatos_f4.json) la evidencia verificable de entorno, artefactos leídos con `usecols`, contrato cumplido frente a `diccionario_f3.csv`, verificación muestral contra `data/raw`, esquema de `copia_graficar` y matriz de vinculación objetivo-pregunta-figura.


In [5]:
# 5) Exportar docs/metadatos_f4.json con la trazabilidad verificada hasta la copia para graficar.
metadatos_f4 = {
    "fase": "F4",
    "etapa": "Contrato de artefactos, verificación contra data/raw y copia para graficar",
    "entorno": {
        "python": platform.python_version(),
        "sistema": platform.system(),
        "entorno_virtual_activo": bool(sys.prefix != sys.base_prefix),
        "paquetes": {
            fila["Paquete"]: {"instalada": fila["Instalada"], "declarada": fila["Declarada"], "coincide": bool(fila["Coincide"])}
            for _, fila in versiones_f4.iterrows()
        },
        "estilo_figuras_rcparams": ESTILO_FIGURAS_F4,
    },
    "artefactos": {
        "separador": ";",
        "ofertas": {"filas": int(len(ofertas_f4)), "columnas_archivo": len(cols_archivo_ofertas), "usecols": list(COLUMNAS_OFERTAS_F4)},
        "licitaciones": {"filas": int(len(licitaciones_f4)), "columnas_archivo": len(cols_archivo_lic), "usecols": list(COLUMNAS_LICITACIONES_F4)},
        "diccionario_filas": int(len(diccionario)),
        "parametros_filas": int(len(parametros)),
        "contrato_aprobado": True,
    },
    "verificacion_raw": verificacion_raw,
    "copia_graficar": {
        "filas_comparables": int(len(copia_graficar)),
        "columnas_agregadas": ["RazonMonto", "LogRazonMonto", "SentidoBrecha", "TamanoGanador", "TamanoGanadorOrdinal", "EtiquetaCoincidencia", "BrechaRelativaPct", "TramoEstimado"],
        "eje_ordinal": list(ORDEN_TAMANO),
        "fuera_del_eje_ordinal": ["NoClasificado", "Múltiple"],
        "conteos_por_moneda": {str(k): int(v) for k, v in copia_graficar["MonedaLicitacion"].value_counts().items()},
        "conteos_por_tramo": {str(k): int(v) for k, v in copia_graficar["TramoEstimado"].value_counts(sort=False).items()},
    },
    "vinculacion_objetivos": [
        {
            "objetivo": "Objetivo específico 1 (proporción de coincidencia)",
            "pregunta": "P1 (parte 1): proporción comparable en que estimado == total ganador",
            "figura": "01_coincidencia_y_sentido.png",
            "funciones_comunicacion": ["intervalo_wilson", "resumen_sentido_por_moneda", "graficar_coincidencia_y_sentido"],
        },
        {
            "objetivo": "Objetivo específico 2 (magnitud y sentido por moneda)",
            "pregunta": "P1 (parte 2): magnitud y sentido de la brecha separando por moneda",
            "figura": "01_coincidencia_y_sentido.png / 02_distribucion_log_razon_clp.png",
            "funciones_comunicacion": ["resumen_magnitud_por_moneda", "graficar_distribucion_brecha"],
        },
        {
            "objetivo": "Objetivo específico 3 (comparación por tamaño ganador)",
            "pregunta": "P2: cambio de coincidencia y brecha según tamaño del proveedor ganador",
            "figura": "03_resultados_por_tamano_clp.png",
            "funciones_comunicacion": ["tabla_resumen_por_tamano", "graficar_resultados_por_tamano"],
        },
    ],
}

ruta_metadatos = raiz / "docs/metadatos_f4.json"
ruta_metadatos.write_text(json.dumps(metadatos_f4, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
assert ruta_metadatos.is_file()
print(f"[OK] Metadatos exportados en {ruta_metadatos.relative_to(raiz)} ({ruta_metadatos.stat().st_size:,} bytes)")


[OK] Metadatos exportados en docs\metadatos_f4.json (4,016 bytes)
